##### **Silver Layer**
- ##### *Clean & standardize raw (dbo) tables → silver*
- Trims text (blank → NULL), enforces data types, pads postal codes, drops NULL / duplicate business keys, adds `_load_ts`.

In [ ]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# table -> (business key, column casts). Text columns are always trimmed.
SPEC = {
    "region":             ("RegionID",      {}),
    "geography":          ("GeographyID",   {"GeographyID": "bigint", "Latitude": "double", "Longitude": "double"}),
    "channels":           ("ChannelID",     {"ChannelID": "bigint"}),
    "order_statuses":     ("OrderStatusID", {"OrderStatusID": "bigint"}),
    "customers":          ("CustomerID",    {"Age": "bigint", "SignupDate": "timestamp"}),
    "customer_segments":  ("CustomerID",    {}),
    "customer_addresses": ("CustomerID",    {}),
    "products":           ("ProductID",     {"UnitCostUSD": "double", "StandardPriceUSD": "double", "IsActive": "bigint"}),
    "orders":             ("OrderID",       {"OrderDateKey": "bigint", "CustomerKey": "bigint", "CustomerChannelKey": "bigint",
                                             "GeographyKey": "bigint", "OrderStatusKey": "bigint"}),
    "order_items":        ("OrderItemID",   {"ProductKey": "bigint", "Quantity": "bigint",
                                             "UnitPriceUSD": "double", "GrossSalesUSD": "double", "DiscountUSD": "double",
                                             "ReturnAmountUSD": "double", "NetRevenueUSD": "double",
                                             "ProductCostUSD": "double", "GrossProfitUSD": "double",
                                             "IsReturned": "bigint", "IsCancelled": "bigint", "IsDelivered": "bigint",
                                             "IsRepeatCustomer": "bigint", "CustomerOrderSequence": "bigint"}),
}

In [ ]:
results = []
for table, (key, casts) in SPEC.items():
    raw = spark.table(f"dbo.{table}_raw")
    df = raw
    for name, dtype in df.dtypes:
        if dtype == "string":
            trimmed = F.trim(F.col(name))
            df = df.withColumn(name, F.when(trimmed == "", None).otherwise(trimmed))
    for name, dtype in casts.items():
        df = df.withColumn(name, F.col(name).cast(dtype))
    if "PostalCode" in df.columns:
        df = df.withColumn("PostalCode", F.lpad(F.col("PostalCode").cast("string"), 5, "0"))

    df = (df.filter(F.col(key).isNotNull())
            .dropDuplicates([key])
            .withColumn("_load_ts", F.current_timestamp()))
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{table}")
    results.append((table, raw.count(), spark.table(f"silver.{table}").count()))

summary = spark.createDataFrame(results, "table_name string, raw_rows long, silver_rows long") \
               .withColumn("dropped_rows", F.col("raw_rows") - F.col("silver_rows"))
display(summary)

In [ ]:
# Stop the pipeline here if any silver table came out empty
empty = [r.table_name for r in summary.collect() if r.silver_rows == 0]
if empty:
    raise ValueError(f"Silver standardization failed - empty tables: {empty}")